In [8]:
from ddgs import DDGS

results = DDGS().images(query="seamless wood grain texture",
    max_results=20,
)
print(results)


[{'title': 'Wood grain texture in 4k seamless pattern on Craiyon', 'image': 'https://pics.craiyon.com/2024-01-02/DWxvrWFpRYy-x8bJgDsd7g.webp', 'thumbnail': 'https://tse2.mm.bing.net/th/id/OIP.IV17c6DFjtd1CtKHoQW2fAHaHa?cb=defcachec2&pid=Api', 'url': 'https://www.craiyon.com/image/M3k3WpxmScu-UOYeMWhvtQ', 'height': 1024, 'width': 1024, 'source': 'Bing'}, {'title': 'Wood grain texture 4k seamless on Craiyon', 'image': 'https://pics.craiyon.com/2023-08-01/d1681ca1b9fc44c0a832aeb65618d0b2.webp', 'thumbnail': 'https://tse3.mm.bing.net/th/id/OIP.7WT4LUGgxoNEZDbXywQGHgHaHa?cb=defcachec2&pid=Api', 'url': 'https://www.craiyon.com/image/_mydTO89T1aRc6Y6NEZ3Gg', 'height': 1024, 'width': 1024, 'source': 'Bing'}, {'title': '5 Free Seamless Wood Textures (JPG)', 'image': 'https://unblast.com/wp-content/uploads/2020/04/Seamless-Wood-Texture-5.jpg', 'thumbnail': 'https://tse1.mm.bing.net/th/id/OIP.oUSKFmwqW1eImYyAw4h6HAHaFj?cb=defcachec2&pid=Api', 'url': 'https://unblast.com/5-free-seamless-wood-textu

In [ ]:
import cv2 as cv
import time
import numpy as np
import torch
from ultralytics import YOLO

def main():
    # Standard Windows camera capture
    cap = cv.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open video device.")
        return

    cv.namedWindow("ParcelVision - SegMode", cv.WINDOW_NORMAL)

    model = YOLO("yolo11n-seg.pt") 
    model.to("cuda")
    
    warmup_frames = 10
    measure_frames = 30
    frame_count = 0
    total_inference_time = 0
    fps_counter = 0
    fps_start_time = time.perf_counter()

    CONF_THRESHOLD = 0.3
    ALLOWED_CLASSES = {"chair", "couch", "dining table"}
    label_map = {"couch": "Sofa", "dining table": "Table", "chair": "Chair"}

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        fps_counter += 1
        torch.cuda.synchronize() 
        start = time.perf_counter()
        
        results = model(frame, verbose=False)
        
        torch.cuda.synchronize() 
        end = time.perf_counter()
        inference_ms = (end - start) * 1000
        frame_count += 1

        if warmup_frames < frame_count <= (warmup_frames + measure_frames):
            total_inference_time += inference_ms

        if frame_count == (warmup_frames + measure_frames):
            average = total_inference_time / measure_frames
            print(f"--- Benchmark Complete ---")
            print(f"Avg Inference: {average:.2f} ms | Est. FPS: {1000/average:.2f}")

        result = results[0]
        if result.masks is not None:
            overlay = frame.copy()
            for mask, box in zip(result.masks.xy, result.boxes):
                class_id = int(box.cls[0])
                class_name = model.names[class_id]
                conf = float(box.conf[0])

                if class_name not in ALLOWED_CLASSES or conf < CONF_THRESHOLD:
                    continue

                mask_points = np.array(mask, dtype=np.int32)
                cv.fillPoly(overlay, [mask_points], (0, 255, 0))

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                business_label = label_map.get(class_name, class_name)
                display_text = f"{business_label} {conf*100:.1f}%"
                
                cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv.putText(frame, display_text, (x1, max(y1 - 10, 0)),
                           cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            cv.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)

        cv.imshow("ParcelVision - SegMode", frame)

        if time.perf_counter() - fps_start_time >= 1.0:
            print(f"Full pipeline FPS: {fps_counter}")
            fps_counter = 0
            fps_start_time = time.perf_counter()
        
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()

if __name__ == "__main__":
    main()

c:\Users\arise\anaconda\envs\alpha\lib\site-packages\torchvision\io\image.py:14: UserWarning: Failed to load image Python extension: 'Could not find module 'C:\Users\arise\anaconda\envs\alpha\Lib\site-packages\torchvision\image.pyd' (or one of its dependencies). Try using the full path with constructor syntax.'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Full pipeline FPS: 1
Full pipeline FPS: 7
Full pipeline FPS: 20
--- Benchmark Complete ---
Avg Inference: 25.57 ms | Est. FPS: 39.11
Full pipeline FPS: 29
Full pipeline FPS: 28
Full pipeline FPS: 18
Full pipeline FPS: 14
Full pipeline FPS: 20
Full pipeline FPS: 19
Full pipeline FPS: 20
Full pipeline FPS: 17
Full pipeline FPS: 18
Full pipeline FPS: 17
Full pipeline FPS: 16
Full pipeline FPS: 17
Full pipeline FPS: 19
Full pipeline FPS: 17
Full pipeline FPS: 20
Full pipeline FPS: 17
Full pipeline FPS: 19
Full pipeline FPS: 18
Full pipeline FPS: 19
Full pipeline FPS: 18
Full pipeline FPS: 18
Full pipeline FPS: 18
Full pipeline FPS: 19
Full pipeline FPS: 18
Full pipeline FPS: 17
Full pipeline FPS: 19
Full pipeline FPS: 20
Full pipeline FPS: 22
Full pipeline FPS: 19
Full pipeline FPS: 13
Full pipeline FPS: 7
Full pipeline FPS: 7
Full pipeline FPS: 8
Full pipeline FPS: 8
Full pipeline FPS: 8
Full pipeline FPS: 7
Full pipeline FPS: 8
Full pipeline FPS: 7
Full pipeline FPS: 6
Full pipeline FPS:

KeyboardInterrupt: 

: 